In [4]:
import pandas as pd
import numpy as np
import json
from sklearn.linear_model import LogisticRegression
from fairlearn.metrics import (
    demographic_parity_difference,
    equalized_odds_difference,
    MetricFrame
)
from sklearn.metrics import accuracy_score

# Load original cleaned CSVs — we need real age values, not scaled ones
datasets_info = {
    "diabetes": {
    "csv": "../datasets/processed/diabetes_clean.csv",
    "age_col": "Age",
    "gender_col": "Sex",
    "age_type": "coded"
},
    "heart": {
        "csv": "../datasets/processed/heart_clean.csv",
        "age_col": "age",
        "gender_col": "male"
    },
    "liver": {
        "csv": "../datasets/processed/liver_clean.csv",
        "age_col": "Age of the patient",
        "gender_col": "Gender"
    },
    "kidney": {
        "csv": "../datasets/processed/kidney_clean.csv",
        "age_col": "age",
        "gender_col": None  # no gender column in kidney
    }
}

def fairness_analysis(name):
    print(f"\nProcessing {name}...")

    info = datasets_info[name]

    # Load arrays for model training/prediction
    X_train = np.load(f"../datasets/processed/{name}_X_train.npy")
    X_test  = np.load(f"../datasets/processed/{name}_X_test.npy")
    y_train = np.load(f"../datasets/processed/{name}_y_train.npy")
    y_test  = np.load(f"../datasets/processed/{name}_y_test.npy")

    # Train logistic regression
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Load original CSV to get real age values for test set
    df_orig = pd.read_csv(info["csv"])

    # Recreate the same test split to get age values
    from sklearn.model_selection import train_test_split
    import joblib

    scaler   = joblib.load(f"../models/{name}_scaler.pkl")
    features = joblib.load(f"../datasets/processed/{name}_features.pkl")

    X_full = df_orig[features]
    y_full = df_orig["target"]

    X_scaled_full = scaler.transform(X_full)
    _, X_test_orig, _, y_test_orig = train_test_split(
        X_scaled_full, df_orig,
        test_size=0.2,
        random_state=42,
        stratify=y_full
    )

    # Get real age values from original dataframe
    age_col = info["age_col"]
    real_ages = y_test_orig[age_col].values

    # Create age groups using real age values
    if info.get("age_type") == "coded":
        age_groups = pd.cut(
            real_ages,
            bins=[0, 4, 9, 14],
            labels=["Under 40", "40-60", "Above 60"]
        ).astype(str)
    else:
        age_groups = pd.cut(
            real_ages,
            bins=[0, 40, 60, 200],
            labels=["Under 40", "40-60", "Above 60"]
        ).astype(str)
     

    # Compute fairness metrics
    mf = MetricFrame(
        metrics=accuracy_score,
        y_true=y_test,
        y_pred=y_pred,
        sensitive_features=age_groups
    )

    dpd = demographic_parity_difference(
        y_true=y_test,
        y_pred=y_pred,
        sensitive_features=age_groups
    )

    eod = equalized_odds_difference(
        y_true=(y_test > 0).astype(int),
        y_pred=(y_pred > 0).astype(int),
        sensitive_features=age_groups
    )

    group_acc = mf.by_group.to_dict()
    max_gap   = max(group_acc.values()) - min(group_acc.values())

    if max_gap <= 0.05:
        bias = "Low"
    elif max_gap <= 0.10:
        bias = "Moderate"
    else:
        bias = "High"

    print(f"  Group Accuracy: {group_acc}")
    print(f"  DPD: {dpd:.4f}, EOD: {eod:.4f}, Bias: {bias}")

    result = {
        "age_fairness": {
            "group_accuracy": {k: round(float(v), 3) for k, v in group_acc.items()},
            "demographic_parity_difference": round(float(dpd), 4),
            "equalized_odds_difference": round(float(eod), 4),
            "max_accuracy_gap": round(float(max_gap), 4),
            "bias_risk": bias
        }
    }

    with open(f"../fairness/{name}_fairness.json", "w") as f:
        json.dump(result, f, indent=4)

    print(f"  Saved: ../fairness/{name}_fairness.json")

fairness_analysis("diabetes")
fairness_analysis("heart")
fairness_analysis("liver")
fairness_analysis("kidney")

print("\nAll fairness analysis complete.")


Processing diabetes...
  Group Accuracy: {'40-60': 0.7495893683738988, 'Above 60': 0.7150772330423103, 'Under 40': 0.8519515477792732}
  DPD: 0.5780, EOD: 0.4258, Bias: High
  Saved: ../fairness/diabetes_fairness.json

Processing heart...
  Group Accuracy: {'40-60': 0.666095890410959, 'Above 60': 0.28, 'Under 40': 0.9146341463414634}
  DPD: 0.9539, EOD: 1.0000, Bias: High
  Saved: ../fairness/heart_fairness.json

Processing liver...
  Group Accuracy: {'40-60': 0.6561189477697293, 'Above 60': 0.641925777331996, 'Under 40': 0.6308058753473601}
  DPD: 0.0283, EOD: 0.0383, Bias: Low
  Saved: ../fairness/liver_fairness.json

Processing kidney...
  Group Accuracy: {'40-60': 1.0, 'Above 60': 1.0, 'Under 40': 1.0}
  DPD: 0.4344, EOD: 0.0000, Bias: Low
  Saved: ../fairness/kidney_fairness.json

All fairness analysis complete.
